# AV + earthquake: demo analysis (Phase 6)

1-iteration Sioux Falls runs (1% sample, 3,608 agents): **baseline** vs **damage**
(artery degraded + link closed at 08:30) vs **quake** (damage + D5 destination
redirect). All metrics come from `analyze.py`; this notebook only displays.
Sweep cells (`output/sweep/...`) plug into the same functions.

In [ ]:
import sys
from pathlib import Path
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'scripts' else CWD
sys.path.insert(0, str(ROOT / 'scripts'))
FIG = ROOT / 'output' / 'demo' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import analyze

T_QUAKE = 8.5 * 3600
RUNS = {
    'baseline': str(ROOT / 'output' / 'demo' / 'baseline'),
    'damage': str(ROOT / 'output' / 'demo' / 'damage'),
    'quake': str(ROOT / 'output' / 'demo' / 'quake'),
}
trips = {label: analyze.read_trips(d) for label, d in RUNS.items()}
print({label: len(df) for label, df in trips.items()})

## 1. Run comparison table (Phase 6 validation checklist inputs)

In [ ]:
summary = analyze.compare_runs(RUNS, T_QUAKE)
summary.T

## 2. Redirect outcomes (D5 splits: home / hospital / stay / random)

In [ ]:
redirect_log = analyze.read_redirect_log(RUNS['quake'])
fig, ax = plt.subplots(figsize=(7, 4))
analyze.plot_redirect_outcomes(ax, redirect_log)
fig.tight_layout()
fig.savefig(str(FIG / 'redirect-outcomes.png'))
from IPython.display import Image, display
display(Image(str(FIG / 'redirect-outcomes.png')))
print(redirect_log['outcome'].value_counts().to_dict())

## 3. Travel-time distributions (overall)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
analyze.plot_travel_time_hist(ax, trips)
fig.tight_layout()
fig.savefig(str(FIG / 'travel-times.png'))
from IPython.display import Image, display
display(Image(str(FIG / 'travel-times.png')))
for label, df in trips.items():
    print(label, analyze.trip_travel_stats(df))

## 4. Hospital access (the emergency metric)

In [ ]:
print(analyze.hospital_trip_stats(trips['quake'], redirect_log, T_QUAKE))
fig, ax = plt.subplots(figsize=(8, 4))
analyze.plot_hospital_arrivals(ax, trips['quake'], redirect_log, T_QUAKE)
fig.tight_layout()
fig.savefig(str(FIG / 'hospital-access.png'))
from IPython.display import Image, display
display(Image(str(FIG / 'hospital-access.png')))


## 5. Link-volume shifts (rerouting evidence)

In [ ]:
base_vols = analyze.link_entries(RUNS['baseline'], T_QUAKE)
quake_vols = analyze.link_entries(RUNS['quake'], T_QUAKE)
fig, ax = plt.subplots(figsize=(8, 6))
analyze.plot_link_shift(ax, base_vols, quake_vols)
fig.tight_layout()
fig.savefig(str(FIG / 'link-shift.png'))
from IPython.display import Image, display
display(Image(str(FIG / 'link-shift.png')))
merged = base_vols.merge(quake_vols, on='link', suffixes=('_base', '_quake'))
merged['delta'] = merged['after_quake'] - merged['after_base']
print(merged.sort_values('delta').head(8).to_string(index=False))

## 6. Completion + stuck (mass conservation)

In [ ]:
for label, d in RUNS.items():
    print(label, 'trips:', len(trips[label]), 'stuck:', analyze.count_stuck(d))